# 한국 소설 RAG - 데이터 수집 + Blob Storage 저장

**파이프라인**
```
Google Books API (검색어 + 작가명 수집)
→ 중복 제거 (ISBN 기준)
→ novels_raw.json 로컬 저장
→ Azure Blob Storage 업로드
```


In [ ]:
!pip install requests pandas tqdm python-dotenv azure-storage-blob -q

In [ ]:
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
import requests
import pandas as pd
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient

load_dotenv(override=True)
print("✅ 라이브러리 로드 완료")


## 1. 설정

In [ ]:
# .env 추가 항목
# GOOGLE_BOOKS_API_KEY=your-key
# AZURE_STORAGE_CONNECTION_STRING=your-connection-string
# AZURE_STORAGE_CONTAINER=novels  (없으면 자동 생성)

GOOGLE_KEY        = os.getenv("GOOGLE_BOOKS_API_KEY")
STORAGE_CONN      = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
CONTAINER_NAME    = os.getenv("AZURE_STORAGE_CONTAINER", "novels")
BASE_URL          = "https://www.googleapis.com/books/v1/volumes"
OUTPUT_DIR        = Path("../data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, val in {
    "GOOGLE_BOOKS_API_KEY":              GOOGLE_KEY,
    "AZURE_STORAGE_CONNECTION_STRING":   STORAGE_CONN,
    "AZURE_STORAGE_CONTAINER":           CONTAINER_NAME,
}.items():
    print(f"{'✅' if val else '❌ None'} {name}")


## 2. 수집 함수

In [ ]:
def fetch_books(query: str, start_index: int = 0, max_results: int = 40) -> dict:
    params = {
        "q":            query,
        "langRestrict": "ko",
        "maxResults":   max_results,
        "startIndex":   start_index,
        "printType":    "books",
        "key":          GOOGLE_KEY,
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"[ERROR] {query} / startIndex {start_index} → {e}")
        return {}


def parse_books(data: dict) -> list[dict]:
    items = data.get("items", [])
    novels = []
    for item in items:
        info = item.get("volumeInfo", {})
        title = info.get("title", "").strip()
        if not title:
            continue

        categories = info.get("categories", [])
        category_str = ", ".join(categories)
        if any(kw in category_str for kw in ["Comics", "Juvenile", "manga"]):
            continue

        isbn = ""
        for id_info in info.get("industryIdentifiers", []):
            if id_info.get("type") == "ISBN_13":
                isbn = id_info.get("identifier", "")
                break

        novels.append({
            "id":         isbn or item.get("id", ""),
            "title":      title,
            "author":     ", ".join(info.get("authors", [])),
            "publisher":  info.get("publisher", "").strip(),
            "pub_year":   info.get("publishedDate", "")[:4],
            "category":   category_str or "한국소설",
            "summary":    info.get("description", "").strip(),
            "keywords":   "",
            "isbn":       isbn,
            "img_url":    info.get("imageLinks", {}).get("thumbnail", ""),
            "page_count": info.get("pageCount", 0),
        })
    return novels


## 3. 수집 대상 목록

In [ ]:
SEARCH_QUERIES = [
    "한국소설", "한국문학", "한국 현대소설", "한국 장편소설",
    "한국 단편소설", "한국 역사소설", "한국 성장소설", "한국 가족소설",
    "한국 추리소설", "한국 판타지소설", "한국 SF소설", "한국 미스터리소설",
    "한국 스릴러소설", "한국 범죄소설", "한국 로맨스소설", "한국 청소년소설",
    "한국 여성소설", "한국 심리소설", "한국 고전소설", "한국 근현대소설",
]

AUTHORS = [
    "한강", "김영하", "정유정", "손원평", "김초엽", "최은영",
    "이기호", "편혜영", "구병모", "김애란", "박민규", "천명관",
    "김훈", "황석영", "조남주", "이미예", "김호연", "정세랑",
    "백수린", "장강명", "임솔아", "최진영", "김숨", "강화길",
    "오정희", "은희경", "신경숙", "공지영", "박경리", "이청준",
    "최인훈", "김동인", "이상", "염상섭", "이효석", "현진건",
    "윤흥길", "조세희", "박완서", "최수철", "김원일", "이문열",
    "성석제", "전경린", "김연수", "이승우", "윤대녕", "권여선",
    "하성란", "천운영", "김경욱", "박형서", "김중혁", "이장욱",
]

print(f"검색어 {len(SEARCH_QUERIES)}개 + 작가 {len(AUTHORS)}명")


## 4. 수집 실행

In [ ]:
all_novels = []
seen_ids = set()

def collect(queries: list[str], prefix: str = ""):
    for query in tqdm(queries, desc=prefix):
        for start in range(0, 200, 40):
            data = fetch_books(query, start_index=start, max_results=40)
            novels = parse_books(data)

            if not novels:
                break

            new_count = 0
            for n in novels:
                key = n["isbn"] or n["title"] + n["author"]
                if key and key not in seen_ids:
                    seen_ids.add(key)
                    if not n["summary"]:
                        n["summary"] = n["title"]
                    all_novels.append(n)
                    new_count += 1

            total = data.get("totalItems", 0)
            if start + 40 >= min(total, 200):
                break
            time.sleep(0.2)

print("=== A. 검색어 기반 수집 ===")
collect(SEARCH_QUERIES, prefix="검색어")

print(f"\n=== B. 작가명 기반 수집 ===")
collect([f"inauthor:{a}" for a in AUTHORS], prefix="작가")

print(f"\n✅ 수집 완료: {len(all_novels)}건")


## 5. 중복 제거 및 필터링

In [ ]:
df = pd.DataFrame(all_novels)

# ISBN 기준 중복 제거
df_isbn    = df[df["isbn"] != ""].drop_duplicates(subset=["isbn"])
df_no_isbn = df[df["isbn"] == ""].drop_duplicates(subset=["title", "author"])
df = pd.concat([df_isbn, df_no_isbn]).reset_index(drop=True)

# summary 너무 짧은 거 제거
df = df[df["summary"].str.len() > 20].reset_index(drop=True)

all_novels = df.to_dict("records")

print(f"✅ 최종: {len(all_novels)}건")
print(f"\n=== 출판년도 분포 ===")
print(df["pub_year"].value_counts().head(10))
print(f"\n=== 쪽수 분포 ===")
print(df["page_count"].describe())


## 6. 로컬 저장

In [ ]:
output_path = OUTPUT_DIR / "novels_raw.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_novels, f, ensure_ascii=False, indent=2)
print(f"✅ 로컬 저장: {output_path} ({len(all_novels)}건)")

# CSV도 저장
df.to_csv(OUTPUT_DIR / "novels_raw.csv", index=False, encoding="utf-8-sig")
print(f"✅ CSV 저장: {OUTPUT_DIR / 'novels_raw.csv'}")


## 7. Azure Blob Storage 업로드

In [ ]:
output_path = OUTPUT_DIR / "novels_raw.json"

blob_service = BlobServiceClient.from_connection_string(STORAGE_CONN)

# 컨테이너 없으면 생성
container_client = blob_service.get_container_client(CONTAINER_NAME)
try:
    container_client.create_container()
    print(f"컨테이너 [{CONTAINER_NAME}] 생성")
except Exception:
    print(f"컨테이너 [{CONTAINER_NAME}] 이미 존재")

# JSON 업로드
blob_client = blob_service.get_blob_client(
    container=CONTAINER_NAME,
    blob="novels_raw.json"
)
with open(output_path, "rb") as f:
    blob_client.upload_blob(f, overwrite=True)

print(f"✅ Blob 업로드 완료: {CONTAINER_NAME}/novels_raw.json")

## 8. 확인

In [ ]:
# Blob에 잘 올라갔는지 확인
blobs = list(container_client.list_blobs())
for b in blobs:
    print(f"  {b.name} ({b.size:,} bytes)")
